In [1]:
import re
import ast

In [ ]:
TXT_PATH ="mlp_qat_int8_params.txt"
VH_PATH  ="mlp_params_flat.vh"

weight & bias Bit-Width

In [23]:
wWidth =8
bWidth =1

In [24]:
def extract_block(text, name):
    m = re.search(rf"\[{re.escape(name)}\]\n(.*?)(?=\n\[|\Z)", text, re.S)
    if not m:
        raise ValueError(f"Block [{name}] not found")
    return m.group(1)

In [25]:
def extract_value(block, key):
    m = re.search(rf"^{re.escape(key)} = (.+)$", block, re.M)
    if not m:
        raise ValueError(f"Key '{key}' not found")
    return m.group(1).strip()

In [26]:
def twos_hex(v, width):
    mask = (1 << width) - 1
    x = v & mask
    hex_digits = (width + 3) // 4
    return f"{width}'h{x:0{hex_digits}X}"

In [27]:
def flatten_row_major(mat):
    out = []
    for row in mat:
        out.extend(row)
    return out

In [28]:
def emit_flat_localparam(name, values, width):
    # RTL 쪽에서 [idx*W +: W]로 읽으므로,
    # concatenation은 역순으로 넣어야 index 0이 LSB 쪽에 위치함
    elems = [twos_hex(v, width) for v in reversed(values)]
    joined = ", ".join(elems)
    return f"localparam [{name.upper()}_WW-1:0] {name} = {{ {joined} }};"

In [29]:
with open(TXT_PATH, "r", encoding="utf-8") as f:
    text = f.read()

In [30]:
fc1_block = extract_block(text, "fc1.weight")
fc2_block = extract_block(text, "fc2.weight")

In [31]:
fc1_shape = ast.literal_eval(extract_value(fc1_block, "shape"))
fc2_shape = ast.literal_eval(extract_value(fc2_block, "shape"))

In [32]:
fc1_int = ast.literal_eval(extract_value(fc1_block, "int_repr"))
fc2_int = ast.literal_eval(extract_value(fc2_block, "int_repr"))

In [33]:
# shape check
if tuple(fc1_shape) != (len(fc1_int), len(fc1_int[0])):
    raise ValueError("fc1 shape mismatch")
if tuple(fc2_shape) != (len(fc2_int), len(fc2_int[0])):
    raise ValueError("fc2 shape mismatch")

In [34]:
hid_n = fc1_shape[0]
in_n  = fc1_shape[1]
out_n = fc2_shape[0]

In [35]:
if fc2_shape[1] != hid_n:
    raise ValueError("fc2 input dim must match fc1 output dim")

In [36]:
fc1_flat = flatten_row_major(fc1_int)   # [o][i]
fc2_flat = flatten_row_major(fc2_int)   # [o][i]

In [37]:
fc1_ww = len(fc1_flat) * wWidth
fc1_bw = hid_n * bWidth
fc2_ww = len(fc2_flat) * wWidth
fc2_bw = out_n * bWidth

In [42]:
lines = []
lines.append("// Auto-generated from mlp_qat_int8_params.txt")
lines.append("// Current model structure from file: 8 -> 32 -> 5")
lines.append(f"// fc1 shape = {fc1_shape}, fc2 shape = {fc2_shape}")
lines.append("")
#lines.append(f"localparam integer IN_N_FILE  = {in_n};")
#lines.append(f"localparam integer HID_N_FILE = {hid_n};")
#lines.append(f"localparam integer OUT_N_FILE = {out_n};")
#lines.append("")
lines.append(f"localparam integer fc1wWidth ={fc1_ww};")
lines.append(f"localparam integer fc1bWidth ={fc1_bw};")
lines.append(f"localparam integer fc2wWidth ={fc2_ww};")
lines.append(f"localparam integer fc2bWidth ={fc2_bw};")
lines.append("")

In [ ]:
# weights
elems_fc1 = ", ".join(twos_hex(v, wWidth) for v in reversed(fc1_flat))
lines.append(f"localparam [(fc1wWidth-1):0] fc1_w_flat = {{ {elems_fc1} }};")
lines.append(f"localparam [(fc1bWidth-1):0] fc1_b_flat = {{fc1bWidth{{1'b0}}}};")
lines.append("")

elems_fc2 = ", ".join(twos_hex(v, wWidth) for v in reversed(fc2_flat))
lines.append(f"localparam [(fc2wWidth-1):0] fc2_w_flat = {{ {elems_fc2} }};")
lines.append(f"localparam [(fc2bWidth-1):0] fc2_b_flat = {{fc2bWidth{{1'b0}}}};")
lines.append("")

In [44]:
with open(VH_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

In [45]:
print(f"Generated: {VH_PATH}")
print(f"Model dims from file: IN={in_n}, HID={hid_n}, OUT={out_n}")

Generated: mlp_params_flat.vh
Model dims from file: IN=8, HID=32, OUT=5
